# Day 7: Data Cleaning with Pandas
Chaat Bazaar dummy dataset. Plan: create a deliberately messy copy of the clean data, detect every problem, count it, decide how to handle it, clean it, then validate the cleaned version against the untouched original.

In [1]:
import pandas as pd
import numpy as np

clean = pd.read_csv('data/chaat_bazaar_sales_dummy.csv')
print(clean.shape)
clean.head()

(35, 12)


,Order_Date,Branch,Order_ID,Order_Type,Aggregator,Item_Name,Category,Quantity,Gross_Sales,Discount,Commission,Net_Sales
0,2026-07-03,Barsha,CB-1002,Delivery,Talabat,Pani Puri,Chaat,1,12,0.0,3.0,9.0
1,2026-07-04,Muweilah,CB-1032,Delivery,Careem,Samosa Chaat,Chaat,3,45,0.0,9.9,35.1
2,2026-07-05,Muweilah,CB-1020,Dine-in,Direct,Vada Pav,Snacks,4,40,6.0,0.0,34.0
3,2026-07-06,Karama,CB-1030,Dine-in,Direct,Gulab Jamun,Desserts,5,50,0.0,0.0,50.0
4,2026-07-06,Barsha,CB-1023,Delivery,Talabat,Masala Chai,Beverages,6,48,2.4,11.4,34.2


## Task 2 Step 0: Introduce the mess (into a COPY only)
Injected on purpose: 2 duplicate orders, 2 missing values, 1 negative sale, 1 impossible quantity, inconsistent branch spellings, one date in a different format, one numeric stored as text. The original file is never modified.

In [2]:
messy = clean.copy()

# 2 duplicate orders (full rows repeated)
messy = pd.concat([messy, messy.iloc[[3, 10]]], ignore_index=True)

# 2 missing values
messy.loc[5, 'Branch'] = np.nan
messy.loc[8, 'Net_Sales'] = np.nan

# 1 negative sale
messy.loc[12, 'Net_Sales'] = -messy.loc[12, 'Net_Sales']

# 1 impossible quantity
messy.loc[15, 'Quantity'] = 500

# inconsistent branch spellings
messy.loc[2, 'Branch'] = 'karama'
messy.loc[20, 'Branch'] = 'KARAMA '

# one date in a different format
messy.loc[7, 'Order_Date'] = '15/07/2026'

# one numeric value stored as text
messy['Gross_Sales'] = messy['Gross_Sales'].astype(object)
messy.loc[18, 'Gross_Sales'] = ' AED 45.0 '

messy.to_csv('data/chaat_bazaar_sales_messy.csv', index=False)
print('messy copy saved:', messy.shape)

messy copy saved: (37, 12)


## Step 1 and 2: Detect and count the problems
Reload from disk so detection works on the file as an analyst would receive it, with no memory of what was injected.

In [3]:
df = pd.read_csv('data/chaat_bazaar_sales_messy.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Order_Date   37 non-null     str    
 1   Branch       36 non-null     str    
 2   Order_ID     37 non-null     str    
 3   Order_Type   37 non-null     str    
 4   Aggregator   37 non-null     str    
 5   Item_Name    37 non-null     str    
 6   Category     37 non-null     str    
 7   Quantity     37 non-null     int64  
 8   Gross_Sales  37 non-null     str    
 9   Discount     37 non-null     float64
 10  Commission   37 non-null     float64
 11  Net_Sales    36 non-null     float64
dtypes: float64(3), int64(1), str(8)
memory usage: 3.6 KB


In [4]:
problems = {}

# duplicates (full row)
problems['duplicate_rows'] = int(df.duplicated().sum())

# missing values
problems['missing_values'] = int(df.isna().sum().sum())
print('missing by column:'); print(df.isna().sum()[df.isna().sum() > 0])

# numeric stored as text: Gross_Sales column dtype
problems['text_typed_numeric_cols'] = int(df['Gross_Sales'].dtype == object)

# coerce a numeric view for value checks
gross_num = pd.to_numeric(df['Gross_Sales'], errors='coerce')
problems['unparseable_gross_values'] = int(gross_num.isna().sum() - df['Gross_Sales'].isna().sum())

# negative sales
problems['negative_net_sales'] = int((df['Net_Sales'] < 0).sum())

# impossible quantity (menu orders realistically 1-20)
problems['impossible_quantity'] = int((df['Quantity'] > 20).sum())

# branch spelling variants
valid_branches = {'JVC', 'Muweilah', 'Karama', 'Barsha'}
problems['branch_spelling_variants'] = int((~df['Branch'].isin(valid_branches) & df['Branch'].notna()).sum())
print('branch values found:', sorted(df['Branch'].dropna().unique()))

# date format problems
dates = pd.to_datetime(df['Order_Date'], format='%Y-%m-%d', errors='coerce')
problems['nonstandard_dates'] = int(dates.isna().sum())

pd.Series(problems)

missing by column:
Branch       1
Net_Sales    1
dtype: int64
branch values found: ['Barsha', 'JVC', 'KARAMA ', 'Karama', 'Muweilah', 'karama']


duplicate_rows              2
missing_values              2
text_typed_numeric_cols     0
unparseable_gross_values    1
negative_net_sales          1
impossible_quantity         1
branch_spelling_variants    2
nonstandard_dates           1
dtype: int64

## Step 3: Decisions, problem by problem
- **Duplicate rows**: drop, keep first. Same Order_ID with every field identical is a repeated record, not a second sale. Never delete duplicates blindly though: two orders with the same items but different Order_IDs would be legitimate.
- **Missing Branch**: cannot be safely guessed. Kept, flagged for source verification; excluded from branch-level reporting until resolved.
- **Missing Net_Sales**: recomputable, because Net = Gross - Discount - Commission and the other three exist. Recompute rather than drop.
- **Negative sale**: a refund would legitimately be negative, but this dataset has no refund flag, so treat as sign error, flag it, and exclude from KPI totals pending verification. Do NOT silently flip the sign.
- **Impossible quantity (500)**: physically implausible for a single chaat order. Flag and exclude; likely a keying error (5 or 50), but guessing which is fabrication.
- **Branch spellings**: standardize with strip + title-case mapping to the four known branches.
- **Nonstandard date**: parse with explicit day-first handling for the one deviant value, convert all to ISO.
- **Text-typed numeric**: strip currency text and convert to float. Exactly the SQLite lesson from Day 4, now in Pandas.
- **Rows failing the accounting identity after cleaning**: quarantine. A value that parses is not a value that is right.

In [5]:
cleaned = df.copy()

# 1. drop full duplicates
cleaned = cleaned.drop_duplicates(keep='first').reset_index(drop=True)

# 2. fix text-typed numeric
cleaned['Gross_Sales'] = (cleaned['Gross_Sales'].astype(str)
                          .str.replace('AED', '', regex=False)
                          .str.strip()
                          .replace('nan', np.nan)
                          .astype(float))

# 3. standardize branch names
cleaned['Branch'] = cleaned['Branch'].str.strip().str.title().replace({'Jvc': 'JVC'})

# 4. standardize dates (handle the day-first stray, then ISO)
def parse_date(x):
    try:
        return pd.to_datetime(x, format='%Y-%m-%d')
    except ValueError:
        return pd.to_datetime(x, dayfirst=True)
cleaned['Order_Date'] = cleaned['Order_Date'].apply(parse_date).dt.strftime('%Y-%m-%d')

# 5. recompute missing Net_Sales from the identity
mask = cleaned['Net_Sales'].isna()
cleaned.loc[mask, 'Net_Sales'] = (cleaned.loc[mask, 'Gross_Sales']
                                  - cleaned.loc[mask, 'Discount']
                                  - cleaned.loc[mask, 'Commission']).round(2)

# 6. flag rows needing source verification instead of silently fixing them
cleaned['Flag'] = ''
cleaned.loc[cleaned['Net_Sales'] < 0, 'Flag'] += 'NEGATIVE_SALE;'
# identity check per row: format-fixed values can still be WRONG values.
# The repaired '45.0' gross does not reconcile with its own Discount/Commission/Net,
# which proves the value itself is corrupt, not just its formatting. Quarantine it.
gap = (cleaned['Gross_Sales'] - cleaned['Discount'] - cleaned['Commission'] - cleaned['Net_Sales']).abs() > 0.01
cleaned.loc[gap & cleaned['Net_Sales'].notna(), 'Flag'] += 'IDENTITY_MISMATCH;'
cleaned.loc[cleaned['Quantity'] > 20, 'Flag'] += 'IMPOSSIBLE_QTY;'
cleaned.loc[cleaned['Branch'].isna(), 'Flag'] += 'MISSING_BRANCH;'

analysis_ready = cleaned[cleaned['Flag'] == ''].drop(columns='Flag')
quarantine = cleaned[cleaned['Flag'] != '']
print('cleaned:', cleaned.shape, '| analysis-ready:', analysis_ready.shape, '| quarantined:', len(quarantine))
quarantine[['Order_ID', 'Branch', 'Quantity', 'Net_Sales', 'Flag']]

cleaned: (35, 13) | analysis-ready: (31, 12) | quarantined: 4


,Order_ID,Branch,Quantity,Net_Sales,Flag
5,CB-1016,NaN,4,78.62,MISSING_BRANCH;
12,CB-1004,Muweilah,1,-12.00,NEGATIVE_SALE;IDENTITY_MISMATCH;
15,CB-1019,Barsha,500,9.00,IMPOSSIBLE_QTY;
18,CB-1033,Muweilah,1,9.36,IDENTITY_MISMATCH;


## Step 5: Validate the cleaned version
Cleaning is only done when it is proven done.

In [6]:
checks = {}
checks['row_count_back_to_35'] = len(cleaned) == 35
checks['no_duplicates'] = not cleaned.duplicated().any()
checks['gross_is_numeric'] = cleaned['Gross_Sales'].dtype == float
checks['branches_standardized'] = set(cleaned['Branch'].dropna().unique()) <= {'JVC', 'Muweilah', 'Karama', 'Barsha'}
checks['dates_iso'] = pd.to_datetime(cleaned['Order_Date'], format='%Y-%m-%d', errors='coerce').notna().all()
checks['net_sales_no_missing'] = cleaned['Net_Sales'].notna().all()
checks['identity_holds_on_analysis_rows'] = ((analysis_ready['Gross_Sales'] - analysis_ready['Discount']
    - analysis_ready['Commission'] - analysis_ready['Net_Sales']).abs() < 0.01).all()

# reconcile against the untouched original, excluding the 3 quarantined orders
orig = pd.read_csv('data/chaat_bazaar_sales_dummy.csv')
orig_comparable = orig[~orig['Order_ID'].isin(quarantine['Order_ID'])]
checks['totals_match_original_on_comparable_rows'] = (
    round(analysis_ready['Net_Sales'].sum(), 2) == round(orig_comparable['Net_Sales'].sum(), 2))

pd.Series(checks)

row_count_back_to_35                        True
no_duplicates                               True
gross_is_numeric                            True
branches_standardized                       True
dates_iso                                   True
net_sales_no_missing                        True
identity_holds_on_analysis_rows             True
totals_match_original_on_comparable_rows    True
dtype: bool

# Task 3: Python Analyst Challenge
Same questions previously answered in SQL, now in Pandas, then reconciled against SQL on the same clean data. Analysis below uses the original clean dataset so results are comparable with Days 3-6.

In [7]:
d = pd.read_csv('data/chaat_bazaar_sales_dummy.csv')

# 1. Total company net sales
total_net = round(d['Net_Sales'].sum(), 2)
print('1. Total net sales:', total_net)

1. Total net sales: 1386.75


In [8]:
# 2. Sales by branch
sales_by_branch = d.groupby('Branch')['Net_Sales'].sum().round(2).sort_values(ascending=False)
sales_by_branch

Branch
Karama      617.40
Muweilah    463.45
JVC         212.62
Barsha       93.28
Name: Net_Sales, dtype: float64

In [9]:
# 3. Sales by aggregator
d.groupby('Aggregator')['Net_Sales'].sum().round(2).sort_values(ascending=False)

Aggregator
Talabat    566.70
Careem     342.11
Direct     328.10
Noon       149.84
Name: Net_Sales, dtype: float64

In [10]:
# 4. Average order value by branch
d.groupby('Branch')['Net_Sales'].mean().round(2).sort_values(ascending=False)

Branch
Karama      51.45
Muweilah    42.13
JVC         30.37
Barsha      18.66
Name: Net_Sales, dtype: float64

In [11]:
# 5. Top 5 products by revenue
d.groupby('Item_Name')['Net_Sales'].sum().round(2).sort_values(ascending=False).head(5)

Item_Name
Paneer Tikka    385.98
Pav Bhaji       181.81
Samosa Chaat    136.50
Falooda         117.20
Masala Chai     114.26
Name: Net_Sales, dtype: float64

In [12]:
# 6. Gross -> discount -> commission -> net reconciliation
recon = pd.Series({'Gross': d['Gross_Sales'].sum(), 'Discounts': -d['Discount'].sum(),
                   'Commission': -d['Commission'].sum(), 'Net (derived)': d['Gross_Sales'].sum() - d['Discount'].sum() - d['Commission'].sum(),
                   'Net (column)': d['Net_Sales'].sum()}).round(2)
print(recon)
assert abs(recon['Net (derived)'] - recon['Net (column)']) < 0.01, 'identity broken'
print('identity holds')

Gross            1822.00
Discounts        -112.40
Commission       -322.85
Net (derived)    1386.75
Net (column)     1386.75
dtype: float64
identity holds


In [13]:
# 7. Branch contribution %
(d.groupby('Branch')['Net_Sales'].sum() / d['Net_Sales'].sum() * 100).round(2).sort_values(ascending=False)

Branch
Karama      44.52
Muweilah    33.42
JVC         15.33
Barsha       6.73
Name: Net_Sales, dtype: float64

In [14]:
# 8. Highest-value order
d.loc[d['Net_Sales'].idxmax(), ['Order_ID', 'Branch', 'Item_Name', 'Net_Sales']]

Order_ID          CB-1021
Branch             Karama
Item_Name    Paneer Tikka
Net_Sales           126.0
Name: 13, dtype: object

## Reconcile Python vs SQL
Load the same CSV into SQLite, run the SQL versions, compare programmatically. They must agree without forcing.

In [15]:
import sqlite3
con = sqlite3.connect(':memory:')
d.to_sql('sales', con, index=False)

sql_total = con.execute('SELECT ROUND(SUM(Net_Sales),2) FROM sales').fetchone()[0]
sql_branch = pd.read_sql('SELECT Branch, ROUND(SUM(Net_Sales),2) AS Net FROM sales GROUP BY Branch', con).set_index('Branch')['Net']
sql_top = con.execute('SELECT Item_Name FROM sales GROUP BY Item_Name ORDER BY SUM(Net_Sales) DESC LIMIT 1').fetchone()[0]

print('total  | pandas:', total_net, '| sql:', sql_total, '| match:', total_net == sql_total)
print('branch | match:', (sales_by_branch.round(2).sort_index() == sql_branch.round(2).sort_index()).all())
print('top product | pandas:', d.groupby('Item_Name')['Net_Sales'].sum().idxmax(), '| sql:', sql_top)

total  | pandas:

 1386.75 | sql: 1386.75 | match: True
branch | match: True
top product | pandas: Paneer Tikka | sql: Paneer Tikka
